# 第74章 共享单车需求与运力调度

使用 UCI Bike Sharing 17,379 条小时级租赁记录，分析通勤峰谷、天气冲击和注册/临时用户差异，并建立无泄漏需求预测基线。

## 项目背景

共享单车运营需要决定何时补车、在哪类天气下预留运力。UCI Bike Sharing Dataset 汇总华盛顿特区 Capital Bikeshare 2011-2012 年逐小时租赁，并匹配天气和日历字段。

## 学习目标

- 理解小时级时间序列结构
- 区分临时与注册用户需求
- 识别工作日和天气下的峰谷
- 避免使用 casual/registered 预测 cnt 的目标泄漏
- 用时间切分评估需求预测


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| dteday/hr | 日期/小时 | 需求时间 |
| workingday/holiday | 工作日/节假日 | 日历变量 |
| weathersit | 天气等级 | 1好至4差 |
| temp/atemp/hum/windspeed | 归一化气象指标 | 连续变量 |
| casual/registered | 临时/注册用户数 | cnt 的组成，不可作预测特征 |
| cnt | 总租赁量 | 预测目标 |

## 数据质量检查清单

- instant 是否唯一
- 日期小时是否重复
- cnt 是否等于 casual+registered
- 时间是否连续及是否存在缺口
- 归一化气象字段范围


## 项目任务

1. 结构审计与时间索引
2. 分析小时和工作日需求
3. 比较用户类型
4. 分析天气条件
5. 用时间切分训练需求模型
6. 输出补车建议与限制


## 项目交付物

- 一份可复现的分析 Notebook
- 清洗规则与关键指标表
- 至少一张支持结论的图表
- 结论、限制和下一步建议

## 阶段检查点

- [ ] 问题和数据字典完成
- [ ] 质量检查和清洗记录完成
- [ ] 核心指标或图表完成
- [ ] 结论与限制完成

## 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 1. 加载与结构审计

验证目标构成关系，并建立真正的时间顺序。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv(f"{base_url}/datasets/bike_sharing_hour.csv", parse_dates=["dteday"])
df["timestamp"]=df.dteday+pd.to_timedelta(df.hr, unit="h")
df = df.sort_values("timestamp")
print("形状:", df.shape, " 时间:", df.timestamp.min(), "至", df.timestamp.max())
print("时间重复:", df.timestamp.duplicated().sum(), " 目标构成错误:", (df.cnt!=df.casual+df.registered).sum())
print(df[["temp", "atemp", "hum", "windspeed", "cnt"]].describe().round(2))


## 2. 通勤峰谷与用户结构

注册用户通常体现通勤规律，临时用户更受休闲场景影响；分开看比总量更有运营价值。


In [ ]:
profile = df.groupby(["workingday", "hr"])[["casual", "registered", "cnt"]].mean()
print("工作日需求最高小时:\n", profile.loc[1].nlargest(5, "cnt").round(1))
print("非工作日需求最高小时:\n", profile.loc[0].nlargest(5, "cnt").round(1))
fig, ax=plt.subplots(figsize=(9,4))
profile.loc[1, ["casual", "registered"]].plot(ax=ax, title="工作日：临时与注册用户小时需求")
ax.set_ylabel("平均租赁量"); plt.tight_layout(); plt.show()


## 3. 天气与运力风险

天气不是随机分配的，比较用于排班情景而非因果断言。


In [ ]:
weather = df.groupby("weathersit").agg(hours=("cnt", "size"), mean_demand=("cnt", "mean"), p90_demand=("cnt", lambda x:x.quantile(.9)), registered_share=("registered", lambda x:x.sum()/df.loc[x.index, "cnt"].sum()))
print(weather.round(2))
heat = df.pivot_table(index="hr", columns="workingday", values="cnt", aggfunc="mean")
print("各场景P90:", df.groupby(["workingday", "weathersit"]).cnt.quantile(.9).round(0).to_dict())


## 4. 时间切分需求预测

按时间保留最后 20% 做测试，不随机打乱未来；明确排除 casual 和 registered 两个目标组成字段。


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

features = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"]
split = int(len(df)*.8); train, test=df.iloc[:split], df.iloc[split:]
cat = ["season", "mnth", "hr", "weekday", "weathersit"]; num=[c for c in features if c not in cat]
prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat), ("num", "passthrough", num)])
model = Pipeline([("prep", prep), ("model", HistGradientBoostingRegressor(max_iter=180, random_state=74))])
model.fit(train[features], train.cnt); pred=model.predict(test[features])
baseline = np.repeat(train.cnt.tail(24*28).mean(), len(test))
print("时间测试区间:", test.timestamp.min(), "至", test.timestamp.max())
print("模型 MAE:", round(mean_absolute_error(test.cnt, pred),1), " 均值基线 MAE:", round(mean_absolute_error(test.cnt, baseline),1))


## 5. 调度建议

预测必须进入库存、站点容量和调度成本体系后才能落地。


In [ ]:
work_peak = profile.loc[1].cnt.idxmax(); off_peak=profile.loc[0].cnt.idxmax()
print(f"1. 工作日全网峰值约在 {work_peak}:00，非工作日约在 {off_peak}:00，补车应在峰值前完成。")
print("2. 以工作日×天气等级的P90作为初始运力情景，并用滚动时间窗回测。")
print("3. 当前数据没有站点库存、OD流向和调度成本，只能做全网需求预测，不能直接生成调度路线。")


## 结论与表达

- 用户类型拆分揭示通勤与休闲需求差异。
- 预测 cnt 时使用 casual/registered 会造成目标泄漏。
- 时间切分比随机切分更接近未来预测。
- 全网需求还不是站点级调度方案。


## 项目验收清单

- 验证 cnt 构成关系
- 能识别工作日峰值
- 模型排除泄漏字段
- 使用时间测试集并与基线比较

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

使用 UCI Bike Sharing 17,379 条小时级租赁记录，分析通勤峰谷、天气冲击和注册/临时用户差异，并建立无泄漏需求预测基线。


### 你已经完成

- 理解小时级时间序列结构
- 区分临时与注册用户需求
- 识别工作日和天气下的峰谷
- 避免使用 casual/registered 预测 cnt 的目标泄漏
- 用时间切分评估需求预测


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 结构审计与时间索引 |
| 步骤 2 | 分析小时和工作日需求 |
| 步骤 3 | 比较用户类型 |
| 步骤 4 | 分析天气条件 |
| 步骤 5 | 用时间切分训练需求模型 |
| 步骤 6 | 输出补车建议与限制 |


### 质量与结论提醒

- instant 是否唯一
- 日期小时是否重复
- cnt 是否等于 casual+registered
- 用户类型拆分揭示通勤与休闲需求差异。
- 预测 cnt 时使用 casual/registered 会造成目标泄漏。
- 时间切分比随机切分更接近未来预测。
- 全网需求还不是站点级调度方案。


### 项目交付检查

- [ ] 验证 cnt 构成关系
- [ ] 能识别工作日峰值
- [ ] 模型排除泄漏字段
- [ ] 使用时间测试集并与基线比较


### 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
